# Code Execution Agent

An LLM can write Python — but only a code execution agent can verify that the code actually runs correctly. `CodeInterpreterTool` runs each snippet in a subprocess with no access to the host filesystem or network, captures stdout, stderr, generated files, matplotlib plots, and pandas DataFrames, and returns them all as structured JSON.

**What you'll build:** an agent that can answer data analysis questions by writing Python, executing it safely, and interpreting the results.

**Time:** ~20 min | **Difficulty:** Intermediate

**What you'll learn:**
- How `CodeInterpreterTool` isolates code in a subprocess
- What `timeout` and `memory_limit_mb` guard against
- How to parse the structured JSON output (stdout, stderr, files, plots, dataframes)
- How to build a data analysis agent that iterates on failing code

## Prerequisites

You need:
- An **OpenAI API key** (`OPENAI_API_KEY`)
- The `synapsekit` package plus `pandas` and `matplotlib` for data analysis examples

In [ ]:
!pip install synapsekit pandas matplotlib -q

In [ ]:
import os

# Set your API key here
# os.environ['OPENAI_API_KEY'] = 'sk-...'

## Step 1: Import and configure the tool

`timeout` limits wall-clock execution time. `memory_limit_mb` caps the subprocess RSS to prevent runaway memory allocation. Both limits kill the subprocess cleanly and return a `ToolResult` with `error` set.

In [ ]:
import asyncio
import json
from synapsekit.agents import CodeInterpreterTool, FunctionCallingAgent
from synapsekit.llms.openai import OpenAILLM

code_tool = CodeInterpreterTool(
    timeout=10.0,        # abort after 10 seconds — prevents infinite loops
    memory_limit_mb=256, # cap at 256 MB — prevents OOM from large datasets
)

print(f"Tool name: {code_tool.name}")
print(f"Tool description: {code_tool.description[:80]}...")

## Step 2: Understand the output format

`CodeInterpreterTool` returns a JSON string in `ToolResult.output`. The keys are:

| Key | Type | Description |
|---|---|---|
| `stdout` | string | Everything printed to stdout |
| `stderr` | string | Tracebacks and warnings |
| `files` | array | `{path, size}` for each file written |
| `plots` | array | `{path, size}` for each matplotlib figure saved |
| `dataframes` | array | `{name, repr}` for each pandas DataFrame in scope |

In [ ]:
async def run_code(code: str) -> dict:
    result = await code_tool.run(code=code)
    if result.is_error:
        return {"error": result.error}
    return json.loads(result.output)

# Test: run a simple computation
output = asyncio.run(run_code("print(sum(range(100)))"))
print("stdout:", output.get("stdout", "").strip())

## Step 3: Build a data analysis agent

The system prompt should explicitly tell the agent to write executable Python and to interpret the output after execution. Without this guidance, the LLM tends to answer from training data rather than running code.

In [ ]:
agent = FunctionCallingAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=[code_tool],
    system_prompt=(
        "You are a data analysis assistant. Always answer quantitative questions "
        "by writing Python code and executing it with the code_interpreter tool. "
        "Parse the JSON output to extract stdout and any dataframe reprs. "
        "If code fails with an error in stderr, fix it and retry once."
    ),
    max_iterations=6,
)

## Step 4: Ask a calculation question

The agent will write Python, execute it, and interpret the output.

In [ ]:
async def analyze(question: str) -> str:
    return await agent.run(question)

answer = asyncio.run(analyze("What is the sum of squares of all prime numbers below 50?"))
print(answer)

## Step 5: Handle multi-step analysis

For questions that require building up state across multiple code cells — loading data, transforming it, then visualizing — instruct the agent to write self-contained scripts that load and process data in a single execution.

In [ ]:
DATA_ANALYSIS_PROMPT = """
You are a data analysis assistant. Follow these rules:
1. Always write self-contained Python scripts — load data and perform analysis in one block.
2. Print a summary of results to stdout so they appear in the output.
3. If matplotlib is used, call plt.savefig('output.png') before plt.show().
4. Interpret the captured stdout and dataframe reprs in your final answer.
"""

analysis_agent = FunctionCallingAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=[CodeInterpreterTool(timeout=15.0, memory_limit_mb=512)],
    system_prompt=DATA_ANALYSIS_PROMPT,
    max_iterations=8,
)

answer = asyncio.run(analysis_agent.run(
    "Generate a list of 20 random integers between 1 and 100 using seed 42, "
    "then compute mean, median, and standard deviation."
))
print(answer)

## Step 6: Parse plots and file artifacts

After a run, check whether the agent wrote plot files or other artifacts by examining `agent.memory.steps`.

In [ ]:
def extract_artifacts(agent: FunctionCallingAgent) -> list[dict]:
    artifacts = []
    for step in agent.memory.steps:
        try:
            payload = json.loads(step.observation)
            artifacts.extend(payload.get("plots", []))
            artifacts.extend(payload.get("files", []))
        except (json.JSONDecodeError, AttributeError):
            pass
    return artifacts

artifacts = extract_artifacts(analysis_agent)
if artifacts:
    print(f"Artifacts: {[a['path'] for a in artifacts]}")
else:
    print("No file artifacts produced in this run.")

## Complete Working Example

A self-contained script that builds a code execution agent, streams step events showing code and stdout for three questions, and collects any artifacts produced.

In [ ]:
import asyncio
import json
from synapsekit.agents import (
    ActionEvent,
    CodeInterpreterTool,
    FinalAnswerEvent,
    FunctionCallingAgent,
    ObservationEvent,
)
from synapsekit.llms.openai import OpenAILLM


def build_agent() -> FunctionCallingAgent:
    return FunctionCallingAgent(
        llm=OpenAILLM(model="gpt-4o-mini"),
        tools=[CodeInterpreterTool(timeout=10.0, memory_limit_mb=256)],
        system_prompt=(
            "You are a Python data analysis assistant. "
            "Always execute code to answer quantitative questions. "
            "Print results to stdout. If code fails, fix and retry once."
        ),
        max_iterations=6,
    )


async def main() -> None:
    agent = build_agent()

    questions = [
        "What is the sum of squares of all prime numbers below 50?",
        (
            "Generate a list of 20 random integers between 1 and 100 using seed 42, "
            "then compute mean, median, and standard deviation."
        ),
        "Write Python to create a simple bar chart of [3, 7, 2, 9, 5] and save it as chart.png.",
    ]

    for question in questions:
        print(f"\nQ: {question}")
        print("-" * 60)

        async for event in agent.stream_steps(question):
            if isinstance(event, ActionEvent):
                code = str(event.tool_input)
                # Show only the first 3 lines of code to keep output readable
                preview = "\n".join(code.splitlines()[:3])
                print(f"[code]\n{preview}\n...")
            elif isinstance(event, ObservationEvent):
                try:
                    payload = json.loads(event.observation)
                    if payload.get("stdout"):
                        print(f"[stdout] {payload['stdout'].strip()}")
                    if payload.get("stderr"):
                        print(f"[stderr] {payload['stderr'].strip()[:100]}")
                    if payload.get("plots"):
                        print(f"[plots]  {[p['path'] for p in payload['plots']]}")
                except json.JSONDecodeError:
                    print(f"[obs] {event.observation[:100]}")
            elif isinstance(event, FinalAnswerEvent):
                print(f"\nAnswer: {event.answer}")

    # Inspect artifacts produced across all questions
    artifacts = []
    for step in agent.memory.steps:
        try:
            payload = json.loads(step.observation)
            artifacts.extend(payload.get("files", []))
            artifacts.extend(payload.get("plots", []))
        except (json.JSONDecodeError, AttributeError):
            pass

    if artifacts:
        print(f"\nArtifacts produced: {[a['path'] for a in artifacts]}")


asyncio.run(main())

## Next Steps

- [Multi-Tool Orchestration](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/multi-tool-orchestration) — combine code execution with web search and database queries
- [SQL Database Agent](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/sql-database-agent) — query a database and pass results to the code interpreter for analysis
- [Tool Error Handling and Retries](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/tool-error-handling) — build retry logic for code that fails on the first attempt